# Obligatorio Steam: exploración y limpieza con Spark

## 1. Sesión de Spark y rutas del Data Lake
Abrimos la sesión de Spark y definimos las rutas de las zonas en HDFS.

In [2]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("obligatorio_steam").getOrCreate()

# Zonas del Data Lake
LANDING   = "/lnd/steam"            # datos crudos, tal como los dejó NiFi
REFINADO  = "/rfn/steam/refinado"   # datos limpios en Parquet
MODELO    = "/rfn/steam/modelo"     # tablas del modelo estrella (Hive apunta acá)
ANALITICA = "/anl/steam"            # resultados de las preguntas 4 y 5

print("Spark:", spark.version)

Spark: 3.5.0


## 2. Lectura de los datos de la zona landing
Leemos los cuatro archivos que dejó NiFi en /lnd/steam y verificamos la cantidad de registros de cada uno.

In [3]:
archivos = {
    "games":       "steam_games",
    "users_items": "australian_users_items",
    "reviews":     "australian_user_reviews",
    "bundles":     "bundle_data",
}

crudos = {nombre: spark.read.json(f"{LANDING}/{archivo}.json.gz")
          for nombre, archivo in archivos.items()}

for nombre, df in crudos.items():
    print(f"{nombre:12s} filas: {df.count():>8,}   columnas: {len(df.columns)}")

games        filas:   32,135   columnas: 16
users_items  filas:   88,310   columnas: 5
reviews      filas:   25,799   columnas: 3
bundles      filas:      615   columnas: 7


## 3. Estructura de las tablas
Revisamos las columnas de cada tabla y el tipo de dato que Spark dedujo para cada una.

In [4]:
for nombre, df in crudos.items():
    print(f"========== {nombre} ==========")
    df.printSchema()

========== games ==========
root
 |-- app_name: string (nullable = true)
 |-- developer: string (nullable = true)
 |-- discount_price: double (nullable = true)
 |-- early_access: boolean (nullable = true)
 |-- genres: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- id: string (nullable = true)
 |-- metascore: string (nullable = true)
 |-- price: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- reviews_url: string (nullable = true)
 |-- sentiment: string (nullable = true)
 |-- specs: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- title: string (nullable = true)
 |-- url: string (nullable = true)

========== users_items ==========
root
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- item_id: string (nullable = true)
 |    |    |-- it

## 4. Exploración del catálogo de juegos
Miramos algunas filas del catálogo y revisamos los valores de precio que no son números.

In [6]:
games = crudos["games"]

games.select("id", "title", "developer", "price", "release_date",
             "genres", "metascore", "sentiment").show(5, truncate=35)

+------+-----------------------+----------------+------------+------------+-----------------------------------+---------+---------------+
|    id|                  title|       developer|       price|release_date|                             genres|metascore|      sentiment|
+------+-----------------------+----------------+------------+------------+-----------------------------------+---------+---------------+
|761140|    Lost Summoner Kitty|       Kotoshiro|        4.99|  2018-01-04|[Action, Casual, Indie, Simulati...|     NULL|           NULL|
|643980|              Ironbound|Secret Level SRL|Free To Play|  2018-01-04|[Free to Play, Indie, RPG, Strat...|     NULL|Mostly Positive|
|670290|Real Pool 3D - Poolians|    Poolians.com|Free to Play|  2017-07-24|[Casual, Free to Play, Indie, Si...|     NULL|Mostly Positive|
|767400|             弹炸人2222|        彼岸领域|        0.99|  2017-12-07|        [Action, Adventure, Casual]|     NULL|           NULL|
|773570|                   NULL|         

In [7]:
(games
 .filter(F.col("price").isNotNull() & F.col("price").cast("double").isNull())
 .groupBy("price")
 .count()
 .orderBy(F.desc("count"))
 .show(30, truncate=False))

+-----------------------------+-----+
|price                        |count|
+-----------------------------+-----+
|Free                         |905  |
|Free to Play                 |520  |
|Free To Play                 |462  |
|Free Mod                     |4    |
|Free Demo                    |3    |
|Third-party                  |2    |
|Play for Free!               |2    |
|Play Now                     |2    |
|Free Movie                   |1    |
|Play WARMACHINE: Tactics Demo|1    |
|Free HITMAN™ Holiday Pack    |1    |
|Free to Try                  |1    |
|Starting at $449.00          |1    |
|Play the Demo                |1    |
|Starting at $499.00          |1    |
|Install Theme                |1    |
|Install Now                  |1    |
|Free to Use                  |1    |
+-----------------------------+-----+



### Valores nulos y validación de la clave primaria

In [8]:
total = games.count()

fila = games.select([F.sum(F.col(c).isNull().cast("int")).alias(c)
                     for c in games.columns]).first()

for c in games.columns:
    print(f"{c:15s} nulos: {fila[c]:>6,}   ({fila[c] / total:.1%})")

app_name        nulos:      2   (0.0%)
developer       nulos:  3,299   (10.3%)
discount_price  nulos: 31,910   (99.3%)
early_access    nulos:      0   (0.0%)
genres          nulos:  3,283   (10.2%)
id              nulos:      2   (0.0%)
metascore       nulos: 29,458   (91.7%)
price           nulos:  1,377   (4.3%)
publisher       nulos:  8,052   (25.1%)
release_date    nulos:  2,067   (6.4%)
reviews_url     nulos:      2   (0.0%)
sentiment       nulos:  7,182   (22.3%)
specs           nulos:    670   (2.1%)
tags            nulos:    163   (0.5%)
title           nulos:  2,050   (6.4%)
url             nulos:      0   (0.0%)


In [9]:
print("Juegos:", total, "| ids distintos:", games.select("id").distinct().count())

games.groupBy("id").count().filter("count > 1").show()

Juegos: 32135 | ids distintos: 32133
+------+-----+
|    id|count|
+------+-----+
|  NULL|    2|
|612880|    2|
+------+-----+



In [10]:
(games
 .filter(F.col("id").isNull() | (F.col("id") == "612880"))
 .select("id", "app_name", "title", "developer", "price", "release_date", "genres", "url")
 .show(truncate=40))

+------+----------------------------------------+----------------------------------------+----------------------------------------+-----+------------+-------------------+----------------------------------------+
|    id|                                app_name|                                   title|                               developer|price|release_date|             genres|                                     url|
+------+----------------------------------------+----------------------------------------+----------------------------------------+-----+------------+-------------------+----------------------------------------+
|  NULL|                                    NULL|                                    NULL|                                    NULL|19.99|        NULL|               NULL|          http://store.steampowered.com/|
|612880|        Wolfenstein II: The New Colossus|        Wolfenstein II: The New Colossus|                           Machine Games|59.99|  2017-10-26|  

In [11]:
games.filter(F.col("id") == "200260").select("id", "app_name").show()

+------+--------------------+
|    id|            app_name|
+------+--------------------+
|200260|Batman: Arkham Ci...|
+------+--------------------+



### Hallazgos y decisiones sobre el catálogo

- **Clave primaria:** hay 32.135 filas y 32.133 `id` distintos.
  - Dos filas tienen `id` nulo. Una no tiene ningún dato y se descarta. La otra es *Batman: Arkham City GOTY*; su URL indica el `id` 200260, que ya existe en otra fila, por lo que es un duplicado y también se descarta.
  - El `id` 612880 (*Wolfenstein II*) aparece dos veces con los mismos datos; se conserva una sola fila.
  - El catálogo limpio queda con 32.132 juegos.
- **Columnas casi vacías:** `discount_price` (99,3 % nulos) y `metascore` (91,7 %) no se usan en el análisis.
- **Nombre del juego:** se usa `app_name`, que está completo, en lugar de `title` (6,4 % nulos).
- **Desarrollador y géneros:** `developer` (10,3 %) y `genres` (10,2 %) tienen nulos; esos juegos quedan fuera de los análisis por desarrollador y por género.
- **Precio:** mezcla números con textos. Regla de limpieza:
  - si es un número, se mantiene;
  - si el texto contiene "free" (sin distinguir mayúsculas), el precio es 0 y el juego se marca como gratuito;
  - si dice "Starting at $X", se toma el valor X;
  - el resto ("Third-party", "Play Now", "Install Now", etc.) queda como precio desconocido.
- **"Free to Play" como género:** aparece dentro de `genres` aunque es un modelo de negocio; se tiene en cuenta en las preguntas por género.

## 5. Exploración de las bibliotecas de usuarios
Revisamos la tabla de usuarios y sus juegos: duplicados, usuarios sin juegos y consistencia de la cantidad declarada.

In [12]:
ui = crudos["users_items"]

ui.select("user_id", "steam_id", "items_count",
          F.size("items").alias("juegos_en_lista")).show(5)

+-----------------+-----------------+-----------+---------------+
|          user_id|         steam_id|items_count|juegos_en_lista|
+-----------------+-----------------+-----------+---------------+
|76561197970982479|76561197970982479|        277|            277|
|          js41637|76561198035864385|        888|            888|
|        evcentric|76561198007712555|        137|            137|
|       Riot-Punch|76561197963445855|        328|            328|
|            doctr|76561198002099482|        541|            541|
+-----------------+-----------------+-----------+---------------+
only showing top 5 rows



In [13]:
filas = ui.count()
print("Filas:", filas, "| user_id distintos:", ui.select("user_id").distinct().count())
print("Filas idénticas repetidas:", filas - ui.dropDuplicates().count())

Filas: 88310 | user_id distintos: 87626
Filas idénticas repetidas: 657


In [14]:
print("Usuarios con la lista de juegos vacía:", ui.filter(F.size("items") == 0).count())
print("items_count distinto del largo de la lista:",
      ui.filter(F.col("items_count") != F.size("items")).count())

Usuarios con la lista de juegos vacía: 16806
items_count distinto del largo de la lista: 0


In [15]:
sin_copias = ui.dropDuplicates()

repetidos = (sin_copias.groupBy("user_id").count()
             .filter("count > 1").select("user_id"))

(sin_copias.join(repetidos, "user_id")
 .select("user_id", "items_count", F.size("items").alias("juegos_en_lista"))
 .orderBy("user_id")
 .show(10))

+-----------------+-----------+---------------+
|          user_id|items_count|juegos_en_lista|
+-----------------+-----------+---------------+
|76561198050680344|        125|            125|
|76561198050680344|        125|            125|
|76561198063371165|         45|             45|
|76561198063371165|         45|             45|
|76561198064484479|        336|            336|
|76561198064484479|        336|            336|
|76561198064956087|         55|             55|
|76561198064956087|         55|             55|
|76561198072861800|          7|              7|
|76561198072861800|          7|              7|
+-----------------+-----------+---------------+
only showing top 10 rows



In [16]:
(sin_copias.join(repetidos, "user_id")
 .select("user_id", "steam_id", "user_url",
         F.aggregate("items", F.lit(0).cast("long"),
                     lambda total, juego: total + juego["playtime_forever"]).alias("minutos_totales"),
         F.aggregate("items", F.lit(0).cast("long"),
                     lambda total, juego: total + juego["playtime_2weeks"]).alias("minutos_2semanas"))
 .orderBy("user_id")
 .show(10, truncate=False))

+-----------------+-----------------+----------------------------------------------------+---------------+----------------+
|user_id          |steam_id         |user_url                                            |minutos_totales|minutos_2semanas|
+-----------------+-----------------+----------------------------------------------------+---------------+----------------+
|76561198050680344|76561198050680344|http://steamcommunity.com/profiles/76561198050680344|240175         |4411            |
|76561198050680344|76561198050680344|http://steamcommunity.com/profiles/76561198050680344|240236         |4472            |
|76561198063371165|76561198063371165|http://steamcommunity.com/profiles/76561198063371165|99563          |2257            |
|76561198063371165|76561198063371165|http://steamcommunity.com/profiles/76561198063371165|99563          |2257            |
|76561198064484479|76561198064484479|http://steamcommunity.com/profiles/76561198064484479|285731         |781             |
|7656119

### Hallazgos y decisiones sobre las bibliotecas de usuarios

- **Usuarios repetidos:** hay 88.310 filas y 87.626 `user_id` distintos (684 filas de más).
  - 657 son copias exactas y se eliminan.
  - En 27 casos el mismo usuario aparece con datos distintos. El `steam_id` y la URL coinciden; solo cambian los minutos jugados, con diferencias muy chicas. Son dos registros de la misma biblioteca tomados en momentos distintos. Se conserva el de más minutos totales y, si empatan, el de más minutos en las últimas dos semanas, ya que las horas jugadas solo aumentan con el tiempo.
- **Usuarios sin juegos:** 16.806 usuarios tienen la biblioteca vacía. Se mantienen en la tabla de usuarios, pero no generan registros de usuario-juego.
- **Consistencia:** `items_count` coincide en todos los casos con la cantidad de juegos de la lista.

## 6. Exploración de las reseñas
Separamos las reseñas de cada usuario en filas individuales y revisamos fechas, duplicados y su relación con el catálogo.

In [17]:
rv = crudos["reviews"]

resenas = (rv.select("user_id", F.explode("reviews").alias("r"))
             .select("user_id", "r.*"))

print("Filas (usuarios):", rv.count(), "| user_id distintos:", rv.select("user_id").distinct().count())
print("Reseñas:", resenas.count())

resenas.select("user_id", "item_id", "posted", "recommend", "helpful", "funny").show(5, truncate=30)

Filas (usuarios): 25799 | user_id distintos: 25485
Reseñas: 59305
+-----------------+-------+-------------------------+---------+------------------------------+-----+
|          user_id|item_id|                   posted|recommend|                       helpful|funny|
+-----------------+-------+-------------------------+---------+------------------------------+-----+
|76561197970982479|   1250| Posted November 5, 2011.|     true|                No ratings yet|     |
|76561197970982479|  22200|    Posted July 15, 2011.|     true|                No ratings yet|     |
|76561197970982479|  43110|   Posted April 21, 2011.|     true|                No ratings yet|     |
|          js41637| 251610|    Posted June 24, 2014.|     true|15 of 20 people (75%) found...|     |
|          js41637| 227300|Posted September 8, 2013.|     true|0 of 1 people (0%) found th...|     |
+-----------------+-------+-------------------------+---------+------------------------------+-----+
only showing top 5 rows



In [18]:
con_anio = F.col("posted").rlike(r"\d{4}")

resenas.groupBy(con_anio.alias("tiene_anio")).count().show()
resenas.filter(~con_anio).select("posted").show(5, truncate=False)

+----------+-----+
|tiene_anio|count|
+----------+-----+
|      true|49186|
|     false|10119|
+----------+-----+

+------------------+
|posted            |
+------------------+
|Posted February 3.|
|Posted May 20.    |
|Posted July 24.   |
|Posted June 16.   |
|Posted June 11.   |
+------------------+
only showing top 5 rows



In [19]:
print("Reseñas repetidas (mismo usuario y juego):",
      resenas.count() - resenas.dropDuplicates(["user_id", "item_id"]).count())

print("Reseñas sin dato de recomendación:", resenas.filter(F.col("recommend").isNull()).count())

ids_catalogo = games.select(F.col("id").alias("item_id")).distinct()
print("Reseñas de juegos que no están en el catálogo:",
      resenas.join(ids_catalogo, "item_id", "left_anti").count())

Reseñas repetidas (mismo usuario y juego): 874
Reseñas sin dato de recomendación: 0
Reseñas de juegos que no están en el catálogo: 5317


In [20]:
print("Reseñas repetidas que son copias exactas:",
      resenas.count() - resenas.dropDuplicates().count())

Reseñas repetidas que son copias exactas: 874


### Hallazgos y decisiones sobre las reseñas

- **Estructura:** cada fila trae la lista de reseñas de un usuario; con `explode` se obtiene una fila por reseña (59.305 en total).
- **Usuarios repetidos:** hay 25.799 filas y 25.485 `user_id` distintos.
- **Reseñas repetidas:** 874 reseñas aparecen dos veces para el mismo usuario y juego. Todas son copias exactas y se eliminan, por lo que quedan 58.431 reseñas.
- **Fechas sin año:** 10.119 reseñas tienen fecha sin año (por ejemplo, "Posted February 3."), porque Steam omite el año en las reseñas del año en curso. Como no se conoce el año de recolección de los datos, esas fechas quedan como desconocidas en lugar de completarse con un año supuesto. La pregunta 3 no depende de la fecha.
- **Recomendación:** la columna `recommend` no tiene nulos.
- **Integridad con el catálogo:** 5.317 reseñas corresponden a juegos que no están en el catálogo. Se conservan, pero no pueden clasificarse por género en la pregunta 3.